# BIST Hacim Tarayıcısı

**TradingView Scanner API** · Alım ilgisi & satış baskısı tahmini · Hızlı Referans.

> Not: Bu net alış/satış verisi değildir; fiyat yönü × hacim ile piyasa ilgisi / baskısı tahmini.

## Endpoint

```
POST https://scanner.tradingview.com/turkey/scan?label-product=screener-stock
```

**Header'lar:** `Content-Type: application/json` · `Origin: https://www.tradingview.com` · `Referer: https://www.tradingview.com/`

`Origin` & `Referer` zorunlu — yoksa 403 dönebilir.

## İstek (Request)

| Body Alanı | Açıklama |
|------------|----------|
| `columns` | Dönecek alan listesi |
| `filter` | Ana filtre · senaryoyu belirler (change vs 0) |
| `sort` | Sıralama · `{sortBy, sortOrder}` |
| `range` | Sonuç dilimi · `[start, end]` |
| `markets` | Piyasa filtresi · `['turkey']` |
| `filter2` | Tip zarfı · stock + common, pre-ipo hariç |
| `options.lang` | Yanıt dili · `'en'` veya `'tr'` |

## Yanıt (Response)

`{ totalCount, data[] }` → `data`, kriterlere uyan BIST hisselerinin sıralı listesi.

### `data[i]`

| Alan | Açıklama |
|------|----------|
| `s` | Sembol · örn. `'BIST:THYAO'` |
| `d[]` | `columns` sırasına göre değerler dizisi |
| `d[0]` | ticker-view · sembol meta nesnesi |
| `d[1]` | close · son fiyat (TL) |
| `d[2]` | change · yüzde değişim |
| `d[3]` | volume · günlük işlem hacmi |
| `d[4]` | market_cap_basic · piyasa değeri (TL) |

## Senaryolar

- `change ≥ 0` · **Alım ilgisi** · hacme göre azalan, ilk 3
- `change < 0` · **Satış baskısı** · hacme göre azalan, ilk 3

## Operatörler (doğrulanan)

- `egreater` → büyük veya eşit (≥)
- `less` → küçük (<)
- `equal` → eşit (=)
- `has` → listede içerir
- `has_none_of` → listede hiçbirini içermez

## Notlar

- Asimetri: alım (`egreater`) eşitliği dahil eder (change ≥ 0); satış (`less`) strict negatif (change < 0).
- `filter2` zarfı stock-only ekran için sabit: `type=stock`, `typespecs has 'common'`, `pre-ipo` hariç.
- `range = [start, end]` · `[0, 3]` ilk 3, `[0, 50]` ilk 50 sonuç.

## 1. Ortak Ayarlar

In [ ]:
import requests

URL = "https://scanner.tradingview.com/turkey/scan?label-product=screener-stock"

HEADERS = {
    "content-type": "application/json",
    "origin": "https://www.tradingview.com",
    "referer": "https://www.tradingview.com/",
}

FILTER2 = {
    "operator": "and",
    "operands": [
        {"operation": {"operator": "and", "operands": [
            {"expression": {"left": "type", "operation": "equal", "right": "stock"}},
            {"expression": {"left": "typespecs", "operation": "has", "right": ["common"]}},
        ]}},
        {"expression": {"left": "typespecs", "operation": "has_none_of", "right": ["pre-ipo"]}},
    ],
}

## 2. Alım İlgisi · BIST'te hacme göre ilk 3 yükselen hisse

Aşağıdaki body, görseldeki minimal istektir — `filter2` zarfı `FILTER2` ile eklenir.

In [ ]:
body = {
    "columns": ["ticker-view", "close", "change", "volume", "market_cap_basic"],
    "filter": [{"left": "change", "operation": "egreater", "right": 0}],
    "sort": {"sortBy": "volume", "sortOrder": "desc"},
    "range": [0, 3],
    "markets": ["turkey"],
    "options": {"lang": "en"},
    "filter2": FILTER2,
}

r = requests.post(URL, headers=HEADERS, json=body)
r.raise_for_status()
alim = r.json()

print(f"Toplam: {alim['totalCount']}\n")
print(f"{'Sembol':<15} {'Fiyat':>10} {'Değişim':>10} {'Hacim':>18}")
print("-" * 56)
for row in alim["data"]:
    sembol = row["s"]
    _, close, change, volume, _ = row["d"]
    print(f"{sembol:<15} {close:>10.2f} {change:>9.2f}% {volume:>18,.0f}")

## 3. Satış Baskısı

Aynı body, sadece `filter[0].operation: "less"` (change < 0).

In [ ]:
body["filter"] = [{"left": "change", "operation": "less", "right": 0}]

r = requests.post(URL, headers=HEADERS, json=body)
r.raise_for_status()
satis = r.json()

print(f"Toplam: {satis['totalCount']}\n")
print(f"{'Sembol':<15} {'Fiyat':>10} {'Değişim':>10} {'Hacim':>18}")
print("-" * 56)
for row in satis["data"]:
    sembol = row["s"]
    _, close, change, volume, _ = row["d"]
    print(f"{sembol:<15} {close:>10.2f} {change:>9.2f}% {volume:>18,.0f}")

## 4. curl Eşdeğeri · `filter2` stocks-only zarfı dahil

Aynı isteği shell'den atmak için referans.

In [ ]:
%%bash
curl -sS 'https://scanner.tradingview.com/turkey/scan?label-product=screener-stock' \
  -X POST \
  -H 'content-type: application/json' \
  -H 'origin: https://www.tradingview.com' \
  -H 'referer: https://www.tradingview.com/' \
  --data-raw '{
    "columns": ["ticker-view","close","change","volume","market_cap_basic"],
    "filter":  [{"left":"change","operation":"egreater","right":0}],
    "sort":    {"sortBy":"volume","sortOrder":"desc"},
    "range":   [0, 3],
    "markets": ["turkey"],
    "options": {"lang":"en"},
    "filter2": {
      "operator": "and",
      "operands": [
        {"operation":{"operator":"and","operands":[
          {"expression":{"left":"type","operation":"equal","right":"stock"}},
          {"expression":{"left":"typespecs","operation":"has","right":["common"]}}
        ]}},
        {"expression":{"left":"typespecs","operation":"has_none_of","right":["pre-ipo"]}}
      ]
    }
  }' | python3 -m json.tool | head -40